In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
# 1. Initialize Spark Session (Usually pre-configured in Databricks)
spark = SparkSession.builder.appName("TextProcessing").getOrCreate()

In [0]:
spark = SparkSession.builder.appName("TextProcessing").getOrCreate()

In [0]:
# 2. Read the text file into a DataFrame
file_path = "/Volumes/students/default/filestore/sejong.txt"
df = spark.read.text(file_path).toDF("line")

In [0]:
# 3. Split lines into sentences by splitting on '.' and exploding into rows
sentences_df = df.select(F.explode(F.split(F.col("line"), r"\.")).alias("sentence"))
sentences_df = sentences_df.filter(F.trim(F.col("sentence")) != "")

In [0]:
sentence_count = sentences_df.count()
print(f"Total Sentences: {sentence_count}")

Total Sentences: 5


In [0]:
# 5. Filter sentences containing 'Sejong'
filtered_sentences = sentences_df.filter(F.col("sentence").contains("Sejong"))
display(filtered_sentences)

sentence
"Sejong University (SJU; Korean: 세종대학교; Hanja: 世宗大學校) is a private research university located in Seoul, South Korea"
"Founded as the Kyung Sung Humanities Institute, it was renamed in 1978 to its present name in honor of Sejong the Great, the fourth king of the Joseon Dynasty and overseer of the creation of the Korean alphabet Hangul"
"Over the years, the university expanded its academic programs and facilities, including the establishment of its main building in 1987 and the Sejong Museum in 1973"


In [0]:
# 6. Transform to uppercase
uppercase_sentences = sentences_df.select(F.upper(F.col("sentence")).alias("uppercase_sentence"))
display(uppercase_sentences)

uppercase_sentence
"SEJONG UNIVERSITY (SJU; KOREAN: 세종대학교; HANJA: 世宗大學校) IS A PRIVATE RESEARCH UNIVERSITY LOCATED IN SEOUL, SOUTH KOREA"
"IT IS KNOWN FOR ITS STANDING IN HOSPITALITY AND TOURISM MANAGEMENT, DANCING, ANIMATION, RHYTHMIC GYMNASTICS, COMPUTER SCIENCE AND AI"
"FOUNDED AS THE KYUNG SUNG HUMANITIES INSTITUTE, IT WAS RENAMED IN 1978 TO ITS PRESENT NAME IN HONOR OF SEJONG THE GREAT, THE FOURTH KING OF THE JOSEON DYNASTY AND OVERSEER OF THE CREATION OF THE KOREAN ALPHABET HANGUL"
"OVER THE YEARS, THE UNIVERSITY EXPANDED ITS ACADEMIC PROGRAMS AND FACILITIES, INCLUDING THE ESTABLISHMENT OF ITS MAIN BUILDING IN 1987 AND THE SEJONG MUSEUM IN 1973"
"IT HAS DEVELOPED INTO A COMPREHENSIVE UNIVERSITY, OFFERING A WIDE RANGE OF UNDERGRADUATE AND GRADUATE PROGRAMS"


In [0]:
# 7. Calculate total character length
total_length = sentences_df.select(F.length(F.col("sentence"))).agg(F.sum("length(sentence)")).collect()[0][0]
print(f"Total Length: {total_length}")

Total Length: 742


In [0]:
# 8. Word Count Pipeline (Split by whitespace, lowercase, group and count)
word_counts = sentences_df.select(F.explode(F.split(F.lower(F.trim(F.col("sentence"))), r"\s+")).alias("Word")) \
    .filter(F.col("Word") != "") \
    .groupBy("Word") \
    .count() \
    .withColumnRenamed("count", "Count")

In [0]:
# 9. Sort word counts by frequency descending
sorted_word_count = word_counts.orderBy(F.col("Count").desc())
display(sorted_word_count)

Word,Count
the,10
in,6
of,6
and,6
its,4
a,3
university,3
sejong,3
it,3
is,2


Databricks visualization. Run in Databricks to view.

In [0]:
# 10. Save the results as a CSV inside your Unity Catalog Volume
output_volume_path = "/Volumes/students/default/filestore/textProcessing_results"

sorted_word_count.write.mode("overwrite").format("csv").save(output_volume_path)